In [ ]:
from datasets import load_dataset

og_dataset = load_dataset("sentence-transformers/natural-questions", cache_dir="datasets", split="train")
dataset = og_dataset.shuffle(seed=22).select(range(1000))

In [ ]:
from query_processing.missing import missing
from query_processing.missclicks import missclicks
from query_processing.moved import moved
from query_processing.process_query import process_query

configs = [
    {"mapping": missclicks, "ratios": [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2]},
    {"mapping": missing, "ratios": [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2]},
    {"mapping": moved, "ratios": [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2]}
]

process_query(dataset, configs)

In [ ]:
from completions.complete import complete
from completions.evaluate import evaluate
from completions.get_predictions import get_predictions

base_dir= "datasets/raw_datasets"
save_dir = "datasets/predictions"                 

await get_predictions(base_dir, save_dir, complete)

base_dir= "datasets/predictions"
save_dir = "datasets/evaluations"     

await get_predictions(base_dir, save_dir, evaluate)

In [ ]:
import os
from datasets import load_from_disk, Dataset, concatenate_datasets

base_dir= "datasets/evaluations"

all_datasets = []

for base_datasets in os.listdir(base_dir):
    base_datasets_path = f"{base_dir}/{base_datasets}"
    for dataset in os.listdir(base_datasets_path):
        base_dataset_path = f"{base_datasets_path}/{dataset}"
        
        loaded_dataset = load_from_disk(base_dataset_path)
        
        filtered_dataset = loaded_dataset.filter(lambda example: example["json_failed"]==False)
        
        filtered_dataset = filtered_dataset.add_column(
            name="function", column=[base_datasets for _ in range(len(filtered_dataset))])
        filtered_dataset = filtered_dataset.add_column(
            name="ratio", column=[dataset for _ in range(len(filtered_dataset))])

        all_datasets.append(filtered_dataset)
        
concatenated = concatenate_datasets(all_datasets)
concatenated.save_to_disk("datasets/concatenated")

In [ ]:
from datasets import load_from_disk, Dataset, concatenate_datasets

dataset = load_from_disk("datasets/concatenated")
df = dataset.to_pandas()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

In [ ]:
df["ratio_value"] = (
    df["ratio"]
      .replace("baseline", "ratio_0")
      .str.replace("ratio_", "", regex=False)
      .astype(float)
)

In [ ]:
df["correct"] = (df["judge_label"] == "correct").astype(int)

summary = (
    df.groupby(["function", "ratio_value"])
      .agg(
          n=("id", "size"),
          accuracy=("correct", "mean"),
          mean_confidence=("confidence", "mean"),
          std_confidence=("confidence", "std"),
      )
      .reset_index()
)

summary["accuracy"] *= 100
summary.round(2)

In [ ]:
baseline_acc = summary.loc[
    summary["function"] == "baseline",
    "accuracy"
].iloc[0]

plt.figure(figsize=(8,5))

plt.axhline(
    baseline_acc,
    color="black",
    linestyle="--",
    label=f"Baseline ({baseline_acc:.1f}%)"
)

for name, g in summary.query("function != 'baseline'").groupby("function"):
    plt.plot(
        g["ratio_value"],
        g["accuracy"],
        marker="o",
        linewidth=2,
        label=name,
    )

plt.xlabel("Error ratio")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plot = summary.query("function != 'baseline'").copy()
plot["drop"] = baseline_acc - plot["accuracy"]

fig, ax = plt.subplots(figsize=(8,5))

for name, g in plot.groupby("function"):
    ax.plot(
        g["ratio_value"],
        g["drop"],
        marker="o",
        linewidth=2,
        label=name.capitalize()
    )

ax.set_xlabel("Error ratio")
ax.set_ylabel("Accuracy drop (percentage points)")
ax.grid(alpha=.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

baseline_conf = summary.loc[
    summary.function=="baseline",
    "mean_confidence"
].iloc[0]

ax.axhline(
    baseline_conf,
    color="black",
    linestyle="--",
    label=f"Baseline ({baseline_conf:.1f})"
)

for name, g in summary.query("function!='baseline'").groupby("function"):
    ax.plot(
        g["ratio_value"],
        g["mean_confidence"],
        marker="o",
        linewidth=2,
        label=name.capitalize()
    )

ax.set_xlabel("Error ratio")
ax.set_ylabel("Mean confidence")
ax.grid(alpha=.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
robustness = (
    df[df.function != "baseline"]
    .groupby("id")["correct"]
    .sum()
)

fig, ax = plt.subplots(figsize=(7,5))

ax.hist(
    robustness,
    bins=16,
    edgecolor="black"
)

ax.set_xlabel("Number of corrupted datasets answered correctly")
ax.set_ylabel("Number of queries")

plt.tight_layout()
plt.show()

In [ ]:
pivot = summary.pivot(
    index="function",
    columns="ratio_value",
    values="accuracy"
)

fig, ax = plt.subplots(figsize=(7,3))

im = ax.imshow(pivot, aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

plt.colorbar(im, label="Accuracy (%)")

plt.tight_layout()
plt.show()